In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_LLM_MODEL = os.getenv("OPENAI_LLM_MODEL")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT")
PINECONE_INDEX_REGION = os.getenv("PINECONE_INDEX_REGION")
PINECONE_INDEX_CLOUD = os.getenv("PINECONE_INDEX_CLOUD")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")
PINECONE_INDEX_DIMENSION = os.getenv("PINECONE_INDEX_DIMENSION")
PINECONE_INDEX_METRIC = os.getenv("PINECONE_INDEX_METRIC")

In [6]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

pc.create_index(
    name=PINECONE_INDEX_NAME,
    dimension=1536,
    metric=PINECONE_INDEX_METRIC,
    spec=ServerlessSpec(
        region=PINECONE_INDEX_REGION,
        cloud=PINECONE_INDEX_CLOUD
    )
)

{
    "name": "wine-reviews",
    "metric": "cosine",
    "host": "wine-reviews-dob4bs2.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [7]:
wine_index = pc.Index(PINECONE_INDEX_NAME)
wine_index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [8]:
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader(file_path="winemag-data-130k-v2.csv",encoding="utf-8")
docs = loader.load()

docs[0]

Document(metadata={'source': 'winemag-data-130k-v2.csv', 'row': 0}, page_content=": 0\ncountry: Italy\ndescription: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.\ndesignation: Vulkà Bianco\npoints: 87\nprice: \nprovince: Sicily & Sardinia\nregion_1: Etna\nregion_2: \ntaster_name: Kerin O’Keefe\ntaster_twitter_handle: @kerinokeefe\ntitle: Nicosia 2013 Vulkà Bianco  (Etna)\nvariety: White Blend\nwinery: Nicosia")

In [9]:
print(len(docs))
print(max(len(doc.page_content) for doc in docs))

129971
1115


In [12]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY
)

In [13]:
from langchain_pinecone import PineconeVectorStore

BATCH_SIZE = 300

for i in range(0, len(docs), BATCH_SIZE):
    batch = docs[i:i + BATCH_SIZE]
    try:
        PineconeVectorStore.from_documents(
            documents=batch,
            index_name=PINECONE_INDEX_NAME,
            embedding=embeddings
        )

        print(f"Indexed documents {i} to {i + BATCH_SIZE - 1}")
    except Exception as e:
        print(f"Error indexing documents {i} to {i + BATCH_SIZE - 1}: {e}")
        

Indexed documents 0 to 299
Indexed documents 300 to 599
Indexed documents 600 to 899
Indexed documents 900 to 1199
Indexed documents 1200 to 1499
Indexed documents 1500 to 1799
Indexed documents 1800 to 2099
Indexed documents 2100 to 2399
Indexed documents 2400 to 2699
Indexed documents 2700 to 2999
Indexed documents 3000 to 3299
Indexed documents 3300 to 3599
Indexed documents 3600 to 3899
Indexed documents 3900 to 4199
Indexed documents 4200 to 4499
Indexed documents 4500 to 4799
Indexed documents 4800 to 5099
Indexed documents 5100 to 5399
Indexed documents 5400 to 5699
Indexed documents 5700 to 5999
Indexed documents 6000 to 6299
Indexed documents 6300 to 6599
Indexed documents 6600 to 6899
Indexed documents 6900 to 7199
Indexed documents 7200 to 7499
Indexed documents 7500 to 7799
Indexed documents 7800 to 8099
Indexed documents 8100 to 8399
Indexed documents 8400 to 8699
Indexed documents 8700 to 8999
Indexed documents 9000 to 9299
Indexed documents 9300 to 9599
Indexed documents

In [ ]:
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)
query = "달콤한 맛을 가진 와인"
results = vector_store.similarity_search(query, k=5)
for result in results:
    print(f"Wine: {result.metadata['title']}, Description: {result.page_content[:100]}...\n")